[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/065.%20The%203D%20Pallet_Case%20Packing%20Problem/P65-Tier-11_executed.ipynb)

---


# 65. The 3D Pallet/Case Packing Problem

## Tier 11: A Conceptual Programmable-Matter Simulation

### Purpose and scope

This notebook presents a small, illustrative simulation organized around a programmable-matter idea: allocate abstract material units to requested components, apply local interactions, group units into components, and record their transformation. The code demonstrates that sequence and provides visual summaries of its state.

It is **not** a physical molecular-dynamics model and does **not** solve the geometric 3D bin-packing problem. In particular, it does not establish that the component boxes fit inside the bin without overlap. The reported component-volume ratio is only the sum of specified component volumes divided by bin volume; it is not proof of a feasible packing arrangement. The molecular positions and interaction parameters are illustrative, not calibrated physical quantities.

### Modeling assumptions

- The bin dimensions and component dimensions are measured in millimeters.
- Molecules are abstract material units with randomly initialized positions and velocities. The stated signaling radius of 10 nm is converted to millimeters before comparing it with positions.
- The interaction update is a schematic local rule. Its numerical values do not represent validated molecular forces, masses, or material behavior.
- Each requested component is formed from its required number of allocated units. Its displayed position is the mean position of those units, not a geometrically verified placement.
- The 0.74 reference is the familiar packing fraction for equal spheres in an idealized close-packed arrangement. It is not a universal upper bound for rectangular cases and is not an optimization target validated by this simulation.
- Business figures sometimes associated with programmable matter—such as a particular packing percentage, assembly time, or cost reduction—are not established by this model and are not treated as measured results here.

### Process outline

1. Initialize abstract material units in the bin.
2. Check whether the requested components can be supplied from the available units.
3. Allocate units, apply illustrative environmental triggers, and update local interactions.
4. Group the allocated units into components and record their transformed state.
5. Inspect the resulting component-volume ratio and the simulation histories.

The sequence is useful for discussing how a local-interaction concept might be represented in code. It should not be interpreted as evidence that programmable matter, nanobots, or the claimed logistics performance is currently available.

In [ ]:
import random
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Fix both random-number generators so the illustrative initial state is reproducible.
random.seed(42)
np.random.seed(42)

## Model and implementation

### States, local interactions, and the process sequence

A molecule's **state** records its role in the process: initially it is `liquid`, allocation changes it to `transitioning`, and the solidification stage changes it to `solid`. This state tracking is needed to decide which units are eligible for movement and to make the stages of the simulation visible in the data.

For a molecule at position \(\mathbf{x}_i\), the illustrative update considers other molecules of the same allocated component type within signaling radius \(r_s\). The code applies attraction to units of the same material type and repulsion to units of a different type. In vector form, a contribution has direction \((\mathbf{x}_j-\mathbf{x}_i)/d_{ij}\), where \(d_{ij}=\lVert\mathbf{x}_j-\mathbf{x}_i\rVert\); the magnitude is proportional to a configured coefficient divided by \(d_{ij}^2\). The resulting quantity is used as a schematic acceleration in a simple position update. Because the units and coefficients are not physically calibrated, this is a computational illustration rather than a scientific force model.

The code follows the stated sequence: order checking, allocation, trigger generation and application, local self-organization, component grouping, and solidification. A spatial hash is used only to find nearby candidates for the same local-radius interaction rule; it does not choose component placements or replace the process with a bin-packing algorithm.

### Volume ratio

For formed components \(C\), with component dimensions \((l_c,w_c,h_c)\), and a bin of dimensions \((L,W,H)\), the reported volume ratio is

$$
\rho_{\mathrm{volume}} = \frac{\sum_{c\in C} l_c w_c h_c}{LWH}.
$$

Here, `components` is the formed-component list, `dimensions` stores each component's specified dimensions, and `bin_dims` supplies \((L,W,H)\). This ratio measures volume demand relative to bin volume only. It does not account for orientation, placement, or overlap, so it cannot certify a feasible 3D packing.

In [ ]:
class ProgrammableMatterPacking:
    """Illustrate an abstract, staged programmable-matter packing concept.

    This class tracks material-unit states and component formation. It does not
    compute a collision-free arrangement of component boxes in the bin.
    """

    def __init__(self, bin_dims, base_material_capacity=100000):
        if len(bin_dims) != 3 or any(dimension <= 0 for dimension in bin_dims):
            raise ValueError("Bin dimensions must be three positive values in millimeters.")
        if base_material_capacity < 0:
            raise ValueError("Base material capacity cannot be negative.")

        self.bin_dims = tuple(bin_dims)
        self.L, self.W, self.H = self.bin_dims
        self.base_material_capacity = base_material_capacity
        self.molecules = []
        self.n_molecules = 0

        # Convert the stated nanometer radius to the millimeter coordinate system.
        self.signaling_radius = 10.0e-6
        self.assembly_temperature = 298.15
        self.local_interaction_rules = {
            "attraction_force": 1e-12,
            "repulsion_force": 5e-13,
            "alignment_force": 2e-13,
            "signaling_rate": 1e6,
        }

        # This is a reference value, not a validated target for rectangular cases.
        self.target_density = 0.74
        self.packing_tolerance = 0.05
        self.convergence_threshold = 0.01
        self.assembly_history = []
        self.density_history = []
        self.transformation_events = []
        self._position_index = {}
        self._initialize_component_registry()

        print("Illustrative programmable-matter model initialized:")
        print(f"  Bin dimensions: {self.bin_dims} mm")
        print(f"  Base material capacity: {base_material_capacity:,} units")
        print(f"  Reference sphere-packing fraction: {self.target_density:.0%}")
        print(f"  Registered component types: {len(self.component_registry)}")

    def _initialize_component_registry(self):
        """Store component specifications used by the example orders."""
        self.component_registry = {
            "brake_pad": {
                "name": "brake_pad",
                "required_molecules": 500,
                "dimensions": (200, 100, 30),
                "properties": {
                    "friction": 0.7,
                    "heat_resistance": 450,
                    "durability": 0.9,
                },
            },
            "filter": {
                "name": "filter",
                "required_molecules": 300,
                "dimensions": (80, 80, 100),
                "properties": {
                    "filtration_efficiency": 0.95,
                    "flow_rate": 0.8,
                    "lifetime": 1000,
                },
            },
            "gasket": {
                "name": "gasket",
                "required_molecules": 200,
                "dimensions": (150, 150, 5),
                "properties": {
                    "sealing_pressure": 2.0,
                    "compression_set": 0.3,
                    "chemical_resistance": 0.95,
                },
            },
        }

    def initialize_base_molecules(self, n_molecules):
        """Place abstract base units uniformly in the bin coordinate range."""
        if not isinstance(n_molecules, int) or n_molecules < 0:
            raise ValueError("The number of base units must be a non-negative integer.")
        if n_molecules > self.base_material_capacity:
            raise ValueError("Requested initial units exceed the configured capacity.")

        print(f"Initializing {n_molecules:,} abstract material units...")
        self.molecules = []
        for molecule_id in range(n_molecules):
            molecule = {
                "id": molecule_id,
                "position": (
                    random.uniform(0, self.L),
                    random.uniform(0, self.W),
                    random.uniform(0, self.H),
                ),
                "velocity": (
                    random.gauss(0, 1e-9),
                    random.gauss(0, 1e-9),
                    random.gauss(0, 1e-9),
                ),
                "state": "liquid",
                "material_type": "base",
                "energy_level": random.uniform(0.5, 1.0),
            }
            self.molecules.append(molecule)

        self.n_molecules = len(self.molecules)
        self.density_history.clear()
        self.assembly_history.clear()
        self.transformation_events.clear()
        print(f"Initialized {self.n_molecules:,} units in a {self.L * self.W * self.H:,} mm³ bin.")

    def receive_component_orders(self, orders):
        """Check order validity and total material-unit availability."""
        total_molecules_needed = 0
        print(f"\nReceived {len(orders)} component orders:")

        for component_type, quantity in orders:
            if component_type not in self.component_registry:
                print(f"  {component_type}: unknown component type")
                return False
            if not isinstance(quantity, int) or quantity < 0:
                print(f"  {component_type}: quantity must be a non-negative integer")
                return False

            required = self.component_registry[component_type]["required_molecules"]
            molecules_needed = required * quantity
            total_molecules_needed += molecules_needed
            print(f"  {component_type}: {quantity} units, {molecules_needed:,} material units")

        print(f"Total material units needed: {total_molecules_needed:,}")
        print(f"Available material units: {self.n_molecules:,}")
        if total_molecules_needed <= self.n_molecules:
            print("Sufficient base material is available.")
            return True

        print(f"Insufficient base material: short by {total_molecules_needed - self.n_molecules:,} units.")
        return False

    def initiate_self_assembly(self, orders):
        """Run allocation, triggers, local updates, grouping, and solidification."""
        if not self.receive_component_orders(orders):
            raise ValueError("Orders are invalid or exceed the available material units.")

        print("\n" + "=" * 60)
        print("STARTING THE ILLUSTRATIVE SELF-ASSEMBLY SEQUENCE")
        print("=" * 60)
        start_time = time.time()

        print("\nPhase 1: Material allocation")
        allocated_molecules = self._allocate_molecules_for_components(orders)

        print("\nPhase 2: Environmental trigger activation")
        triggers = self._generate_transformation_triggers(orders)
        self._apply_environmental_triggers(triggers)

        print("\nPhase 3: Local interaction updates and grouping")
        formed_components = self._self_organize_packing(allocated_molecules)

        print("\nPhase 4: Component solidification")
        final_components = self._solidify_components(formed_components)

        elapsed_time = time.time() - start_time
        print("\n" + "=" * 60)
        print("ILLUSTRATIVE PROCESS COMPLETED")
        print("=" * 60)
        print(f"Elapsed wall-clock time: {elapsed_time:.2f} seconds")
        print(f"Components formed: {len(final_components)}")
        print(
            "Component-volume ratio (not a placement-feasibility result): "
            f"{self._calculate_packing_density(final_components):.2%}"
        )
        return final_components

    def _allocate_molecules_for_components(self, orders):
        """Assign distinct base units to each valid component order."""
        allocated_molecules = {}
        available_index = 0
        available_molecules = [
            molecule for molecule in self.molecules
            if molecule["material_type"] == "base"
        ]

        for component_type, quantity in orders:
            if component_type not in self.component_registry:
                continue
            required = self.component_registry[component_type]["required_molecules"]
            molecules_needed = required * quantity
            allocated = available_molecules[
                available_index:available_index + molecules_needed
            ]
            if len(allocated) != molecules_needed:
                raise ValueError(f"Insufficient units to allocate {component_type}.")

            available_index += molecules_needed
            allocated_molecules[component_type] = allocated
            for molecule in allocated:
                molecule["material_type"] = component_type
                molecule["state"] = "transitioning"

            print(f"  Allocated {molecules_needed:,} units to {quantity} {component_type}(s).")

        return allocated_molecules

    def _generate_transformation_triggers(self, orders):
        """Create the illustrative trigger sequence for each ordered type."""
        triggers = []
        for component_type, quantity in orders:
            if component_type not in self.component_registry or quantity == 0:
                continue
            triggers.append({
                "type": "temperature",
                "threshold": 350.0,
                "duration": 5.0,
                "affected_components": [component_type],
                "transformation_type": "compression",
            })
            triggers.append({
                "type": "temperature",
                "threshold": 273.15,
                "duration": 10.0,
                "affected_components": [component_type],
                "transformation_type": "solidification",
            })
        return triggers

    def _apply_environmental_triggers(self, triggers):
        """Update abstract unit attributes according to each trigger stage."""
        print(f"Applying {len(triggers)} illustrative triggers:")
        for trigger in triggers:
            affected = set(trigger["affected_components"])
            print(
                f"  {trigger['type']} trigger at {trigger['threshold']:.2f} K "
                f"for {', '.join(trigger['affected_components'])}"
            )
            for molecule in self.molecules:
                if molecule["material_type"] not in affected:
                    continue
                if trigger["transformation_type"] == "compression":
                    molecule["energy_level"] *= 1.2
                elif trigger["transformation_type"] == "solidification":
                    molecule["state"] = "solid"

    def _build_position_index(self, molecules):
        """Index units into radius-sized cells to find local interaction candidates."""
        cell_width = self.signaling_radius
        position_index = defaultdict(list)
        for molecule in molecules:
            cell = tuple(
                int(np.floor(coordinate / cell_width))
                for coordinate in molecule["position"]
            )
            position_index[cell].append(molecule)
        return position_index

    def _self_organize_packing(self, allocated_molecules):
        """Apply local updates, record a bounding-box proxy, and form components."""
        unit_count = sum(len(group) for group in allocated_molecules.values())
        print(f"Updating local interactions for {unit_count:,} allocated units.")
        max_iterations = 100

        for iteration in range(max_iterations):
            all_allocated = [
                molecule
                for group in allocated_molecules.values()
                for molecule in group
            ]
            self._position_index = self._build_position_index(all_allocated)
            self._update_molecular_positions(allocated_molecules)

            current_proxy = self._calculate_emergent_density(allocated_molecules)
            self.density_history.append(current_proxy)
            self.assembly_history.append({
                "iteration": iteration,
                "bounding_box_coverage_proxy": current_proxy,
            })

            if (
                iteration > 10
                and abs(current_proxy - self.density_history[-2])
                < self.convergence_threshold
            ):
                print(
                    f"  Proxy change fell below the threshold at iteration {iteration}; "
                    "this is not evidence of an optimal packing."
                )
                break
            if iteration % 20 == 0:
                print(f"  Iteration {iteration}: coverage proxy = {current_proxy:.6f}")

        return self._form_components_from_molecules(allocated_molecules)

    def _update_molecular_positions(self, allocated_molecules):
        """Update positions using the schematic local interaction rule."""
        dt = 1e-6
        for molecules in allocated_molecules.values():
            for molecule in molecules:
                if molecule["state"] not in ("liquid", "transitioning"):
                    continue

                ax, ay, az = self._calculate_local_force(molecule, molecules)
                old_velocity = molecule["velocity"]
                new_velocity = (
                    old_velocity[0] + ax * dt,
                    old_velocity[1] + ay * dt,
                    old_velocity[2] + az * dt,
                )
                molecule["velocity"] = new_velocity
                position = molecule["position"]
                molecule["position"] = (
                    max(0.0, min(position[0] + new_velocity[0] * dt, self.L)),
                    max(0.0, min(position[1] + new_velocity[1] * dt, self.W)),
                    max(0.0, min(position[2] + new_velocity[2] * dt, self.H)),
                )

    def _calculate_local_force(self, molecule, molecules):
        """Sum schematic attraction or repulsion from nearby indexed units."""
        force = np.zeros(3, dtype=float)
        cell_width = self.signaling_radius
        cell = tuple(
            int(np.floor(coordinate / cell_width))
            for coordinate in molecule["position"]
        )

        # Adjacent cells contain every possible neighbor within one cell width.
        for dx_cell in (-1, 0, 1):
            for dy_cell in (-1, 0, 1):
                for dz_cell in (-1, 0, 1):
                    neighbor_cell = (
                        cell[0] + dx_cell,
                        cell[1] + dy_cell,
                        cell[2] + dz_cell,
                    )
                    for other in self._position_index.get(neighbor_cell, []):
                        if other["id"] == molecule["id"]:
                            continue
                        displacement = (
                            np.asarray(other["position"], dtype=float)
                            - np.asarray(molecule["position"], dtype=float)
                        )
                        distance = float(np.linalg.norm(displacement))
                        if distance <= 0.0 or distance >= self.signaling_radius:
                            continue

                        same_type = molecule["material_type"] == other["material_type"]
                        if same_type:
                            coefficient = self.local_interaction_rules["attraction_force"]
                            force += coefficient * displacement / distance**3
                        else:
                            coefficient = self.local_interaction_rules["repulsion_force"]
                            force -= coefficient * displacement / distance**3

        return tuple(force)

    def _calculate_emergent_density(self, allocated_molecules):
        """Return summed group bounding-box coverage, a non-feasibility proxy."""
        bin_volume = self.L * self.W * self.H
        if bin_volume <= 0:
            return 0.0

        occupied_proxy = 0.0
        for molecules in allocated_molecules.values():
            if not molecules:
                continue
            positions = np.asarray(
                [molecule["position"] for molecule in molecules], dtype=float
            )
            extent = positions.max(axis=0) - positions.min(axis=0)
            occupied_proxy += float(np.prod(extent))
        return occupied_proxy / bin_volume

    def _form_components_from_molecules(self, allocated_molecules):
        """Group each allocated order into components of its specified unit count."""
        components = []
        for component_type, molecules in allocated_molecules.items():
            spec = self.component_registry[component_type]
            groups = self._group_molecules_by_component(
                molecules, spec["required_molecules"]
            )
            for group in groups:
                components.append({
                    "type": component_type,
                    "molecules": group,
                    "spec": spec,
                    "position": self._calculate_component_center(group),
                    "dimensions": spec["dimensions"],
                    "properties": spec["properties"],
                })
        return components

    def _group_molecules_by_component(self, molecules, required_count):
        """Partition allocated units into complete, non-overlapping groups."""
        if required_count <= 0:
            raise ValueError("The required component unit count must be positive.")
        return [
            molecules[start:start + required_count]
            for start in range(0, len(molecules) - required_count + 1, required_count)
        ]

    def _calculate_component_center(self, molecules):
        """Return the mean unit position as an illustrative component marker."""
        positions = np.asarray(
            [molecule["position"] for molecule in molecules], dtype=float
        )
        return tuple(float(value) for value in positions.mean(axis=0))

    def _solidify_components(self, formed_components):
        """Mark formed units as solid and record their abstract transformation."""
        final_components = []
        for component in formed_components:
            for molecule in component["molecules"]:
                molecule["state"] = "solid"
            self.transformation_events.append({
                "component_type": component["type"],
                "position": component["position"],
                "dimensions": component["dimensions"],
                "properties": component["properties"],
                "timestamp": time.time(),
            })
            final_components.append(component)
        return final_components

    def _calculate_packing_density(self, components):
        """Return component volume divided by bin volume, not packing feasibility."""
        if not components:
            return 0.0
        occupied_volume = sum(
            length * width * height
            for component in components
            for length, width, height in [component["dimensions"]]
        )
        bin_volume = self.L * self.W * self.H
        return occupied_volume / bin_volume

    def visualize_programmable_matter_packing(
        self, components, title="Illustrative Component Centers"
    ):
        """Plot component-center markers inside the bin coordinate frame."""
        fig = plt.figure(figsize=(10, 7))
        ax = fig.add_subplot(111, projection="3d")
        self._draw_bin_edges(ax)

        colors = {"brake_pad": "tab:red", "filter": "tab:blue", "gasket": "tab:green"}
        for component_type in sorted({item["type"] for item in components}):
            selected = [item for item in components if item["type"] == component_type]
            centers = np.asarray([item["position"] for item in selected], dtype=float)
            if centers.size:
                ax.scatter(
                    centers[:, 0], centers[:, 1], centers[:, 2],
                    color=colors.get(component_type, "tab:gray"),
                    label=component_type,
                    s=30,
                )

        ax.set_xlabel("Length (mm)")
        ax.set_ylabel("Width (mm)")
        ax.set_zlabel("Height (mm)")
        ax.set_xlim(0, self.L)
        ax.set_ylim(0, self.W)
        ax.set_zlim(0, self.H)
        ax.set_title(
            f"{title}\nComponent centers; no collision-free placement is computed"
        )
        if components:
            ax.legend(title="Component type")
        plt.tight_layout()
        plt.show()
        self._print_programmable_matter_details(components)

    def _draw_bin_edges(self, ax):
        """Draw the bin coordinate frame used by the illustrative positions."""
        vertices = [
            (0, 0, 0), (self.L, 0, 0), (self.L, self.W, 0), (0, self.W, 0),
            (0, 0, self.H), (self.L, 0, self.H),
            (self.L, self.W, self.H), (0, self.W, self.H),
        ]
        edges = [
            (0, 1), (1, 2), (2, 3), (3, 0),
            (4, 5), (5, 6), (6, 7), (7, 4),
            (0, 4), (1, 5), (2, 6), (3, 7),
        ]
        for start, end in edges:
            points = [vertices[start], vertices[end]]
            ax.plot3D(*zip(*points), color="black", alpha=0.35, linewidth=1)

    def _draw_box(self, ax, x, y, z, length, width, height,
                  color="red", alpha=0.7, label=""):
        """Draw a cuboid when a caller supplies a verified placement."""
        vertices = [
            [x, y, z], [x + length, y, z],
            [x + length, y + width, z], [x, y + width, z],
            [x, y, z + height], [x + length, y, z + height],
            [x + length, y + width, z + height], [x, y + width, z + height],
        ]
        faces = [
            [vertices[0], vertices[1], vertices[5], vertices[4]],
            [vertices[2], vertices[3], vertices[7], vertices[6]],
            [vertices[0], vertices[3], vertices[7], vertices[4]],
            [vertices[1], vertices[2], vertices[6], vertices[5]],
            [vertices[4], vertices[5], vertices[6], vertices[7]],
            [vertices[0], vertices[1], vertices[2], vertices[3]],
        ]
        collection = Poly3DCollection(
            faces, alpha=alpha, facecolor=color, edgecolor="black"
        )
        ax.add_collection3d(collection)
        if label:
            ax.text(x, y, z, label)

    def _print_programmable_matter_details(self, components):
        """Report component attributes and clearly qualified volume statistics."""
        volume_ratio = self._calculate_packing_density(components)
        print("\n" + "=" * 60)
        print("ILLUSTRATIVE COMPONENT SUMMARY")
        print("=" * 60)
        print(f"Components formed: {len(components)}")
        print(f"Distinct component types: {len({item['type'] for item in components})}")
        print(f"Component-volume ratio: {volume_ratio:.2%}")
        print("This ratio does not establish non-overlapping placement or feasibility.")

        for index, component in enumerate(components, start=1):
            x, y, z = component["position"]
            length, width, height = component["dimensions"]
            print(
                f"  {component['type']} {index}: "
                f"{length} × {width} × {height} mm; "
                f"illustrative center ({x:.3f}, {y:.3f}, {z:.3f}) mm"
            )
            for property_name, value in component["properties"].items():
                print(f"    {property_name}: {value}")

    def plot_assembly_convergence(self):
        """Plot the recorded bounding-box coverage proxy and a schematic signal."""
        if not self.density_history:
            print("No interaction history is available to plot.")
            return

        iterations = np.arange(len(self.density_history))
        signaling_activity = np.maximum(
            0.1, 1.0 - iterations / max(len(self.density_history), 1)
        )
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        ax1.plot(iterations, self.density_history, color="tab:blue", linewidth=2)
        ax1.set_xlabel("Iteration")
        ax1.set_ylabel("Summed bounding-box coverage proxy")
        ax1.set_title("Illustrative Position-Update History")
        ax1.grid(True, alpha=0.3)

        ax2.plot(iterations, signaling_activity, color="tab:red", linewidth=2)
        ax2.set_xlabel("Iteration")
        ax2.set_ylabel("Schematic signaling activity (relative units)")
        ax2.set_title("Illustrative Signaling Schedule")
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        print("The proxy and signaling schedule are illustrative, not measured performance.")
        print(f"Recorded iterations: {len(self.density_history)}")
        print(f"Initial coverage proxy: {self.density_history[0]:.6f}")
        print(f"Final coverage proxy: {self.density_history[-1]:.6f}")

    def compare_with_traditional_methods(self):
        """Explain why unsupported performance comparisons are not calculated."""
        print("\nPerformance comparison")
        print("This simulation does not measure packing time, inventory cost, or stockouts.")
        print("Such comparisons require defined, validated data and a comparable baseline.")
        print(
            "The computed component-volume ratio is not directly comparable to a "
            "measured space-utilization result."
        )

## Automotive-parts example

The example keeps the stated bin dimensions, material capacity, and component orders. The availability check concerns only the count of abstract material units. It is distinct from geometric feasibility: component dimensions, orientations, and collision-free placements are not optimized or verified here.

The displayed centers are averages of the allocated units' positions. The visualization therefore plots center markers rather than drawing component-sized boxes that could imply a feasible arrangement.

In [ ]:
print("=" * 60)
print("ILLUSTRATIVE AUTOMOTIVE-PARTS SCENARIO")
print("=" * 60)

bin_dims = (1000, 800, 600)
base_material_capacity = 50000
print("Scenario: Automotive-parts distributor")
print(f"Bin dimensions: {bin_dims} mm")
print(f"Base material capacity: {base_material_capacity:,} abstract units")
print("Scope: component formation illustration, not a verified 3D packing solution")

pm_system = ProgrammableMatterPacking(bin_dims, base_material_capacity)
pm_system.initialize_base_molecules(base_material_capacity)

orders = [
    ("brake_pad", 20),
    ("filter", 15),
    ("gasket", 10),
]

if pm_system.receive_component_orders(orders):
    formed_components = pm_system.initiate_self_assembly(orders)
    pm_system.visualize_programmable_matter_packing(
        formed_components,
        "Illustrative Automotive-Parts Component Centers",
    )
    pm_system.plot_assembly_convergence()
    pm_system.compare_with_traditional_methods()
else:
    print("The requested material-unit count is unavailable.")

## Inspecting the stages

A smaller instance makes the allocation, trigger, interaction, grouping, and solidification stages easier to inspect. The same component specifications and local-interaction sequence are used. As in the main example, the resulting component centers and volume ratio do not certify a collision-free packing.

In [ ]:
print("\n" + "=" * 60)
print("STAGE-BY-STAGE MODEL INSPECTION")
print("=" * 60)

test_bin_dims = (200, 150, 100)
test_capacity = 5000
pm_test = ProgrammableMatterPacking(test_bin_dims, test_capacity)
pm_test.initialize_base_molecules(test_capacity)

test_orders = [("brake_pad", 3), ("filter", 2)]
print(f"\nTest instance: {test_bin_dims} mm bin, {test_capacity:,} abstract units")
print(f"Orders: {test_orders}")

if pm_test.receive_component_orders(test_orders):
    print("\nPhase 1: Material allocation")
    allocated = pm_test._allocate_molecules_for_components(test_orders)
    for component_type, molecules in allocated.items():
        mean_energy = np.mean([molecule["energy_level"] for molecule in molecules])
        print(f"  {component_type}: {len(molecules):,} units allocated")
        print(f"    Mean illustrative energy level: {mean_energy:.3f}")

    print("\nPhase 2: Environmental trigger description")
    triggers = pm_test._generate_transformation_triggers(test_orders)
    for trigger in triggers:
        print(
            f"  {trigger['type']} trigger at {trigger['threshold']:.2f} K "
            f"for {', '.join(trigger['affected_components'])}; "
            f"duration {trigger['duration']:.1f} s"
        )
    pm_test._apply_environmental_triggers(triggers)

    print("\nPhase 3: Local interaction updates and grouping")
    formed = pm_test._self_organize_packing(allocated)
    print(f"  Components formed: {len(formed)}")
    print(
        "  Component-volume ratio: "
        f"{pm_test._calculate_packing_density(formed):.3%}"
    )

    print("\nPhase 4: Solidification")
    final = pm_test._solidify_components(formed)
    print(f"  Components recorded as solid: {len(final)}")
    pm_test.visualize_programmable_matter_packing(
        final,
        "Stage-by-Stage Example: Component Centers",
    )
    pm_test.plot_assembly_convergence()
else:
    print("The requested material-unit count is unavailable.")

## Interpretation, limitations, and extensions

### What the example demonstrates

- The order check compares requested material-unit counts with the initialized supply.
- Allocation assigns distinct base units to requested component types.
- Environmental triggers change abstract unit attributes according to the modeled stages.
- Local interactions update positions using a signaling-radius neighborhood.
- Grouping and solidification record the requested component counts when enough units are available.
- The plotted markers show mean unit positions; the convergence plot records a bounding-box coverage proxy and a schematic signaling schedule.

### What it does not demonstrate

The simulation does not establish material behavior, self-assembly in a physical system, packing optimality, collision avoidance, or real-world improvements in time, cost, inventory, or stockouts. A small change in the recorded proxy is only a stopping condition for this illustrative update process; it is not a proof of convergence to an optimal solution. The 0.74 sphere-packing reference does not validate the volume ratio for these rectangular components.

A genuine 3D pallet/case packing study would need, at minimum, explicit component placements and orientations, bin-boundary constraints, and non-overlap checks. Those modeling elements are outside the scope of this conceptual programmable-matter illustration.